# 🚀 Ultimate PostgreSQL Connector - User Guide

Welcome to the official tutorial for our team's `PostgresConnector` package! 

This connector is designed to make database interactions seamless, especially for data engineering and AI tasks. It supports:
* **Fast Data Ingestion:** Optimized bulk inserts and native Upsert (`ON CONFLICT DO UPDATE`).
* **Auto Schema Evolution:** Automatically adds missing columns and infers data types (including `JSONB`).
* **TimescaleDB Integration:** Easily convert tables into hyper-tables for time-series data.
* **pgvector Integration:** Automatically detect and store AI embeddings with vector indexes.

Let's dive into how to use it!

## 1. Prerequisites and Installation

First, ensure you have the necessary libraries installed. Run the cell below if you haven't installed them yet.

#### Dependencies of the package
```
pandas sqlalchemy psycopg2-binary pgvector loguru
````

## 2. Establishing the Connection

To connect to our PostgreSQL database, import the `PostgresConnector` and initialize it with your database credentials. 

*(Note: Replace the dummy credentials with your actual local or staging database info).*

### Initialize

In [ ]:
import pandas as pd
from postgres_connector import PostgresConnector # Make sure postgres_connector.py is in your directory

# Initialize the connector
pg = PostgresConnector(
    host='localhost', 
    database='my_staging_db', 
    username='postgres', 
    password='mysecretpassword', 
    port=5432
)

# Optional: Setup database extensions for TimescaleDB and pgvector (Run once per database)
pg.setup_extensions()

## 3. Core Feature: Smart Upsert

The `upsert_data` method is your go-to function. It will:
1. Create the table if it doesn't exist.
2. Set the Primary Key automatically.
3. Insert new records or update existing ones based on the `primary_key`.

In [ ]:
# 1. Create some sample user data
user_data = {
    'user_id': [1, 2, 3],
    'name': ['Alice', 'Bob', 'Charlie'],
    'status': ['Active', 'Inactive', 'Active']
}
df_users = pd.DataFrame(user_data)

# 2. Upsert data to the 'users' table
pg.upsert_data(
    df=df_users, 
    target_table='users', 
    primary_key='user_id', 
    conflict_strategy='last' # 'last' means update existing records with new values
)

print("Data successfully upserted!")

## 4. Auto Schema Evolution & JSONB Support

What happens if our data source changes and introduces a new column? Or what if we have nested dictionaries?
The connector handles this automatically! It will `ALTER TABLE` to add the new column and map dictionaries to PostgreSQL's `JSONB` type.

In [ ]:
# Notice the new 'age' column and the 'metadata' column containing dictionaries
updated_user_data = {
    'user_id': [2, 4],
    'name': ['Bob', 'Diana'],
    'status': ['Active', 'Active'],
    'age': [30, 28], # New Column!
    'metadata': [{'role': 'admin'}, {'role': 'user'}] # Will be mapped to JSONB!
}
df_users_v2 = pd.DataFrame(updated_user_data)

# Upsert again. Bob's status will be updated, Diana will be inserted, and new columns will be created.
pg.upsert_data(
    df=df_users_v2, 
    target_table='users', 
    primary_key='user_id'
)

# Let's verify the data
result_df = pg.get_data("SELECT * FROM users;")
display(result_df)

## 5. Advanced: TimescaleDB for Time-Series Data

If you are dealing with logs, stock prices, or IoT data, you should use TimescaleDB. It partitions your data by time for blazing-fast queries.

In [ ]:
# 1. Generate time-series data
dates = pd.date_range(start='2026-03-01', periods=5, freq='D')
log_data = pd.DataFrame({
    'timestamp': dates,
    'server_id': ['S1', 'S1', 'S2', 'S1', 'S2'],
    'cpu_usage': [45.5, 50.1, 88.0, 44.2, 90.5]
})

# 2. Upsert the data (using both timestamp and server_id as composite primary keys)
pg.upsert_data(
    df=log_data, 
    target_table='server_metrics', 
    primary_key=['timestamp', 'server_id']
)

# 3. Convert the table into a TimescaleDB Hypertable
# This partitions the data by 1-day chunks behind the scenes.
pg.enable_timescaledb(
    table_name='server_metrics', 
    time_column='timestamp', 
    chunk_time_interval='1 day'
)

## 6. Advanced: pgvector for AI Embeddings

Building a RAG (Retrieval-Augmented Generation) app? The connector automatically detects lists of floats and converts them into Vector types.

In [ ]:
# 1. Create document data with embeddings (lists of floats)
# In real scenarios, these embeddings come from models like OpenAI or HuggingFace.
docs_data = pd.DataFrame({
    'doc_id': [101, 102],
    'text': ['PostgreSQL is a powerful database.', 'Vectors are used in AI.'],
    'embedding': [
        [0.1, 0.5, -0.2], 
        [0.8, -0.1, 0.4]  
    ] 
})

# 2. Upsert the documents
pg.upsert_data(
    df=docs_data, 
    target_table='ai_documents', 
    primary_key='doc_id'
)

# 3. Create an HNSW index on the vector column for lightning-fast similarity search
pg.create_vector_index(
    table_name='ai_documents', 
    vector_column='embedding', 
    index_type='hnsw'
)

print("AI table with vector index created successfully!")

## 7. Clean Up

It's always a good practice to close the database engine connection when you are done.

In [ ]:
# Dispose of the connection pool
pg.dispose()
print("Connection closed. Happy coding!")